# Fleet and Equipment Analysis

## Goal

This notebook examines whether fleet and equipment characteristics (truck make, truck age,
trailer type) are associated with delivery performance, how fuel efficiency and equipment
utilization behave across the fleet, and how diesel prices developed over the three years.

1. What does the fleet look like, and which parts of it are actually used?
2. Do delay and duration differ by truck characteristics or trailer type?
3. Do individual trucks stand out, or is the spread consistent with chance?
4. How does fuel efficiency (MPG) vary across trucks, trailers, and trips?
5. How did diesel prices develop over time?
6. How do utilization, maintenance cost, and downtime develop over time?

## Data Basis

- `trucks`, `trailers`, `fuel_purchases`, `truck_utilization_metrics` - new for this notebook
- `loads`, `trips`, `delivery_events` - already loaded via the pipeline, providing
  `delivery_delay_hours`, `is_delayed_delivery`, `delivery_duration_hours`

*Note on the data: the dataset is described by its source as a realistic simulation.
Earlier notebooks found little variation across time, region, and equipment; this notebook
therefore places particular emphasis on testing whether an effect is real or consistent with
random variation, rather than on the size of the findings.*

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from delivery_pipeline import run_pipeline, PROCESSED_DATA_PATH_FLEET, RAW_DATA_DIR

clean_df = run_pipeline(output_path=PROCESSED_DATA_PATH_FLEET, include_routes=False, include_fleet=True)
clean_df = clean_df.drop(columns=["route_id"])
print(f"Loads used for analysis: {len(clean_df)}")

fuel_df = pd.read_csv(RAW_DATA_DIR / "fuel_purchases.csv")
util_df = pd.read_csv(RAW_DATA_DIR / "truck_utilization_metrics.csv")
print(f"Fuel purchases: {len(fuel_df)}, Utilization records: {len(util_df)}")

sns.set_theme(style="darkgrid")



## 2. Data Quality Checks
### 2.1 How many trips have missing truck, trailer, or driver references?

In [ ]:
display(clean_df[['truck_id','trailer_id','driver_id']].isna().mean())
display((clean_df.truck_id.isna() & clean_df.trailer_id.isna()).sum())

*Data Quality Note: `truck_id`/`trailer_id`/`driver_id` are missing in about 2% of
rows, consistent with the "intentional 2% null rate" the dataset description states.
`trucks` and `trips` are loaded directly from the raw CSVs for the checks below,
rather than via `clean_df.truck_id`. The pipeline drops 486 loads (0.57%) with
inconsistent timestamps; a truck that only appears in those dropped loads would
otherwise look unused even though it has trips. Loading `trips` raw avoids that
edge case. 36 loads are missing both `truck_id` and `trailer_id` simultaneously, roughly
what independent ~2% null rates would predict by chance.*

### 2.2 Which trucks never appear in trips, and why?

In [ ]:
trucks_df = pd.read_csv(RAW_DATA_DIR / "trucks.csv")
trips_df = pd.read_csv(RAW_DATA_DIR / "trips.csv")

unused_trucks = trucks_df[~trucks_df.truck_id.isin(trips_df.truck_id)]
display(unused_trucks.status.value_counts())

display(unused_trucks[unused_trucks.status == "Active"])

trucks_df["used"] = trucks_df.truck_id.isin(trips_df.truck_id)
trucks_df.groupby("status").used.value_counts().unstack()

*Data Quality Note: Of the 120 trucks, 28 never appear in `trips`: 15 with status 
`Maintenance` and 13 with status `Inactive`. This split is exact, all 92 `Active` trucks 
have trips, and none of the `Maintenance`/`Inactive` trucks do. Truck status alone 
perfectly predicts trip presence in this dataset.*